# Project Pipeline: Predicting Weak Momentum Periods
Run top to bottom. All stage-specific helpers are imported from `src/`.

In [1]:
# --- run me first ---
from pathlib import Path
import os, sys
if Path.cwd().name == "notebooks":
    os.chdir("..")
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("working from:", ROOT.name)


working from: project


## Stage 01 - Problem framing
Stakeholder: portfolio manager. Decision: whether a high-risk environment deserves additional review of momentum exposure.

## Stage 04 - Ingestion

In [2]:
from src.ingest import load_bundled_market_data
raw = load_bundled_market_data("data/raw")
print(raw.shape)
raw.head()

(104, 8)


,date,spy_return,spmo_return,excess_return,Mkt_RF,SMB,HML,RF
0,2017-01-31,0.0179,0.0092,-0.0087,0.0194,-0.0116,-0.0276,0.0004
1,2017-02-28,0.0393,0.0398,0.0005,0.0355,-0.0215,-0.0150,0.0004
2,2017-03-31,0.0012,-0.0187,-0.0199,0.0017,0.0113,-0.0333,0.0003
3,2017-04-30,0.0099,0.0260,0.0161,0.0108,0.0066,-0.0205,0.0005
4,2017-05-31,0.0141,0.0186,0.0045,0.0107,-0.0252,-0.0382,0.0006


## Stage 06 - Preprocessing

In [3]:
from src.cleaning import clean_project_data
clean = clean_project_data(raw)
print(clean.isna().sum())
clean.head()

date             0
spy_return       0
spmo_return      0
excess_return    0
Mkt_RF           0
SMB              0
HML              0
RF               0
dtype: int64


,date,spy_return,spmo_return,excess_return,Mkt_RF,SMB,HML,RF
0,2017-01-31,0.0179,0.0092,-0.0087,0.0194,-0.0116,-0.0276,0.0004
1,2017-02-28,0.0393,0.0398,0.0005,0.0355,-0.0215,-0.0150,0.0004
2,2017-03-31,0.0012,-0.0187,-0.0199,0.0017,0.0113,-0.0333,0.0003
3,2017-04-30,0.0099,0.0260,0.0161,0.0108,0.0066,-0.0205,0.0005
4,2017-05-31,0.0141,0.0186,0.0045,0.0107,-0.0252,-0.0382,0.0006


## Stage 07 - Outlier risk

In [4]:
from src.outliers import detect_outliers_iqr
clean["iqr_outlier"] = detect_outliers_iqr(clean["excess_return"])
print("IQR flags:", int(clean["iqr_outlier"].sum()))
clean.loc[clean["iqr_outlier"], ["date","excess_return"]]

IQR flags: 3


,date,excess_return
72,2023-01-31,-0.0675
76,2023-05-31,-0.0597
85,2024-02-29,0.0627


Outliers are flagged, not automatically removed, because extreme financial returns may be real market events.

## Stage 08 - EDA

In [5]:
from src.eda import eda_summary
eda = eda_summary(clean)
eda["summary"].loc[["spy_return","spmo_return","excess_return","Mkt_RF","SMB","HML"]]

,count,mean,std,min,25%,50%,75%,max
spy_return,104.0,0.012634,0.045680,-0.1246,-0.010825,0.01980,0.038025,0.1270
spmo_return,104.0,0.016243,0.048893,-0.0981,-0.012900,0.01755,0.044950,0.1358
excess_return,104.0,0.003610,0.022998,-0.0675,-0.012325,0.00400,0.015500,0.0627
Mkt_RF,104.0,0.010712,0.047541,-0.1337,-0.017050,0.01715,0.034750,0.1360
SMB,104.0,-0.002296,0.028382,-0.0593,-0.023150,-0.00455,0.014300,0.0714
HML,104.0,-0.002569,0.039367,-0.1383,-0.025025,-0.00585,0.016400,0.1286


## Stage 09 - Feature engineering

In [6]:
from src.features import build_features
features = build_features(clean)
features.to_csv("data/processed/model_dataset.csv", index=False)
features[["date","excess_lag1","excess_mean_3m","spy_vol_3m","target_underperform_next"]].tail()

,date,excess_lag1,excess_mean_3m,spy_vol_3m,target_underperform_next
99,2025-04-30,-0.0154,0.008533,0.026058,0
100,2025-05-31,0.0307,0.022167,0.059671,0
101,2025-06-30,0.0512,0.033400,0.038415,0
102,2025-07-31,0.0183,0.025033,0.020496,1
103,2025-08-31,0.0056,0.003400,0.017164,0


## Stage 10 - Modeling

In [7]:
from src.modeling import BASELINE_FEATURES, FACTOR_FEATURES, time_split, fit_classifier, classifier_metrics, save_model
model_df = features.dropna(subset=FACTOR_FEATURES + ["target_excess_next"]).copy()
train, test = time_split(model_df, 0.75)
baseline = fit_classifier(train, BASELINE_FEATURES)
factor_model = fit_classifier(train, FACTOR_FEATURES)
print("baseline:", classifier_metrics(baseline, test, BASELINE_FEATURES))
print("factor augmented:", classifier_metrics(factor_model, test, FACTOR_FEATURES))
save_model(baseline, "model/model.pkl")

baseline: {'roc_auc': 0.6833333333333333, 'accuracy': 0.6153846153846154, 'precision': 0.3, 'recall': 0.5, 'f1': 0.375, 'brier': 0.2388072888949801}
factor augmented: {'roc_auc': 0.48333333333333334, 'accuracy': 0.5769230769230769, 'precision': 0.2727272727272727, 'recall': 0.5, 'f1': 0.35294117647058826, 'brier': 0.24531010953870216}


PosixPath('model/model.pkl')

## Stage 13 - Productization check

In [8]:
from pathlib import Path
print("model exists:", Path("model/model.pkl").exists())
print("API entry point exists:", Path("app.py").exists())

model exists: True
API entry point exists: True


## Stage 14 - Monitoring artifacts

In [9]:
for p in ["docs/monitoring_plan.md","docs/handoff_plan.md"]:
    print(p, Path(p).exists())

docs/monitoring_plan.md True
docs/handoff_plan.md True


## Stage 15 - Orchestration / CLI step

In [10]:
from src.run_step import build_feature_step
build_feature_step("data/processed/model_dataset.csv", "data/processed/features_cli.csv")
print(Path("data/processed/features_cli.csv").exists())

True


## Stage 16 - Lifecycle review

In [11]:
required = [
"docs/lifecycle_framework_guide.md","docs/project_summary.md","README.md",
"docs/orchestration_plan.md","docs/monitoring_plan.md","reports/stakeholder_report.md"
]
{p: Path(p).exists() for p in required}

{'docs/lifecycle_framework_guide.md': True,
 'docs/project_summary.md': True,
 'README.md': True,
 'docs/orchestration_plan.md': True,
 'docs/monitoring_plan.md': True,
 'reports/stakeholder_report.md': True}